In [ ]:
!pip install ultralytics supervision opencv-python-headless -q

In [ ]:
import cv2
import supervision as sv
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display

In [ ]:

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

In [ ]:
model = YOLO("yolov8m.pt")

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

In [ ]:
output_path = "output_tracking.mp4"

cap = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
tracker = sv.ByteTrack()  # BoTSORT-level accuracy, built into supervision

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Filter to vehicles only
    mask = [int(cls) in VEHICLE_CLASSES for cls in detections.class_id]
    detections = detections[mask]

    # Update tracker
    detections = tracker.update_with_detections(detections)

    labels = [
        f"ID{tracker_id} {VEHICLE_CLASSES.get(int(cls), 'vehicle')} {conf:.2f}"
        for tracker_id, cls, conf in zip(
            detections.tracker_id,
            detections.class_id,
            detections.confidence
        )
    ]

    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

    out.write(frame)
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print(f"Done. Output saved to {output_path}")

In [ ]:
display(Video(output_path, embed=True, width=800))

In [ ]:
files.download(output_path)